# VisA PCB1–3 공유 K-shot 재실행

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고 **런타임 → 모두 실행**을 누릅니다.
Drive 마운트 권한을 승인하면 팀 커밋 `30e32db`와 커밋에 포함된 공용 분할표로 실행됩니다.
기본 실험은 `K1_seed0`이며 PCB별 결함 종류 4개에서 실제 불량을 1장씩 사용합니다.
각 방법은 PCB 3종 × 결함 4종 × 40장, 총 480장과 클래스 ID 마스크를 저장합니다.

copy-paste는 팀 코드 그대로 실행합니다. cTransGAN은 팀 공용 K 분할표를 기존 학습기에 연결하는
아래 코드를 사용합니다. 예전 A팀 분할표는 이미지·마스크 해시 대조에만 사용합니다.
GAN 출력은 정상 이미지 위에 생성된 32×32 패치를 원본 결함 좌표에 합성합니다.
JPEG 저장, 원본 해상도, 클래스별 샘플 수와 manifest 열은 팀의 copy-paste 규격에 맞춥니다.
마스크는 합성 영역을 나타냅니다. 검증·시험 이미지가 소스에 섞이면 검사가 실패합니다.

출력 및 cTransGAN 체크포인트는 `MyDrive/DL_TP/runs/shared_30e32db/`에 저장합니다.
세션이 끊기면 같은 설정으로 다시 실행해 Drive의 체크포인트에서 이어갑니다.
기본 설정은 K=1 파일럿입니다. 이어서 K=2와 K=4를 실행하려면 첫 코드 셀의
`K_VALUES = (1, 2, 4)`로 설정합니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

from google.colab import drive

COMMIT = '30e32dba9ba7fa76b410aef5a642c28abf8f8578'
REPO_URL = 'https://github.com/Junho-YEO-maker/26_2_DeepLearning.git'
REPO = Path('/content/26_2_DeepLearning_30e32db')
AUG = REPO / 'Augmentation/visa_pcb'
DRIVE = Path('/content/drive/MyDrive/DL_TP')
RUN = DRIVE / 'runs/shared_30e32db'
K_VALUES = (1,)
SPLIT_SEED = 0
PER_TYPE = 40

drive.mount('/content/drive')
assert DRIVE.is_dir(), f'프로젝트 Drive 경로를 확인하세요: {DRIVE}'
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=True)
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(AUG / 'requirements-stage1.txt')], check=True)
os.chdir(AUG)
sys.path.insert(0, str(AUG))

In [ ]:
import hashlib
import tarfile

LOCAL_DATA = Path('/content/visa_pcb123_30e32db')
LOCAL_ARCHIVE = Path('/content/VisA_20220922.tar')
SOURCE_ARCHIVE = DRIVE / 'data/VisA_20220922.tar'
ARCHIVE_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
BOARDS = ('pcb1', 'pcb2', 'pcb3')
ready = LOCAL_DATA / '.archive_verified'

def file_sha256(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

if not (ready.is_file() and ready.read_text() == ARCHIVE_SHA
        and all((LOCAL_DATA / p / 'image_anno.csv').is_file() for p in BOARDS)):
    assert SOURCE_ARCHIVE.is_file(), f'VisA 원본 tar 경로를 확인하세요: {SOURCE_ARCHIVE}'
    if not LOCAL_ARCHIVE.is_file() or file_sha256(LOCAL_ARCHIVE) != ARCHIVE_SHA:
        temporary = LOCAL_ARCHIVE.with_suffix('.part')
        shutil.copyfile(SOURCE_ARCHIVE, temporary)
        assert file_sha256(temporary) == ARCHIVE_SHA, 'VisA archive SHA-256 mismatch'
        temporary.replace(LOCAL_ARCHIVE)
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with tarfile.open(LOCAL_ARCHIVE) as archive:
        members = [m for m in archive if Path(m.name).parts[0] in BOARDS]
        archive.extractall(LOCAL_DATA, members=members, filter='data')
    assert all((LOCAL_DATA / p / 'image_anno.csv').is_file() for p in BOARDS)
    ready.write_text(ARCHIVE_SHA)
assert all((LOCAL_DATA / pcb / 'Data').is_dir() for pcb in BOARDS)
print('원본 데이터:', LOCAL_DATA)

In [ ]:
import csv
from collections import Counter

from src.data.shared_splits import PCBS, TYPES, check_manifest, read_rows

def publish(manifest):
    rows = read_rows(manifest)
    counts = Counter((r['category'], r['defect_type']) for r in rows)
    assert counts == {(p, t): PER_TYPE for p in PCBS for t in TYPES}
    for row in rows:
        for key in ('image', 'mask'):
            path = Path(row[key])
            assert not path.is_absolute() and (REPO / path).is_file()
    print(check_manifest(manifest), flush=True)
    dest = RUN / manifest.parent.relative_to(REPO)
    shutil.copytree(manifest.parent, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns('manifest.csv'))
    temporary = dest / 'manifest.csv.tmp'
    shutil.copyfile(manifest, temporary)
    temporary.replace(dest / 'manifest.csv')
    print('Drive 저장 완료:', dest, len(rows), '장')

for module in ('tests.smoke_seeds', 'tests.smoke_k_copypaste'):
    subprocess.run([sys.executable, '-m', module], check=True)
for k in K_VALUES:
    subprocess.run(
        [sys.executable, '-m', 'src.generate.k_copypaste', '--k', str(k), '--seed', str(SPLIT_SEED),
         '--per-type', str(PER_TYPE), '--data-root', str(LOCAL_DATA)],
        check=True,
    )
    publish(REPO / f'Dataset/generated/copypaste/K{k}_seed{SPLIT_SEED}/manifest.csv')

### cTransGAN

Colab GPU에서 기존 모델과 32×32 패치 학습기를 재사용합니다. PCB별로 분류기 200 업데이트와 GAN
4,000 업데이트를 수행합니다. T4는 배치 8, A100은 배치 32이며 둘 다 AMP를 사용합니다.
각 조건의 실제 학습 시간, GPU 메모리 및 패치 개수는 해당 체크포인트의 `progress.json`에 기록됩니다.

In [ ]:
"""Run the existing cTransGAN on the teammate's shared K-shot data."""
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from scipy.ndimage import distance_transform_edt
import torch
import yaml

from src.data.access import checked_manifest
from src.data.gan_patches import PatchDataset
from src.data.prepare import sha, write_csv
from src.data.shared_splits import (
    PCBS, PROJECT_ROOT, TYPES, TYPE_ID, SharedSplits, check_manifest, read_rows,
)
from src.generate.k_copypaste import FIELDS
from src.models.copy_paste import atomic_json
from src.models.ctransgan import build_models
from src.seeds import condition_seed, sample_rng
from src.train.ctransgan import device_for, train, training_signature

METHOD = 'ctransgan'


def restrict_training(splits, audit_dir, dest, pcb, k, seed):
    # The archived split provides verified file/mask/pixel hashes.
    # Only the shared K table determines which images enter training.
    audit = {r['file_path']: r for r in checked_manifest(audit_dir, pcb)}
    forbidden = read_rows(splits.splits / 'test.csv') + read_rows(splits.splits / 'val_normal.csv')
    forbidden_pixels = set()
    for row in forbidden:
        if row['category'] == pcb:
            old = audit[row['image'].removeprefix(splits.prefix)]
            assert old['image_sha256'] == row['image_sha256']
            forbidden_pixels.add(old['pixel_sha256'])

    approved = []
    for row in splits.defect_rows(k, seed, pcb):
        old = audit[row['image'].removeprefix(splits.prefix)]
        assert old['image_sha256'] == row['image_sha256']
        assert old['mask_path'] == row['mask'].removeprefix(splits.prefix)
        assert old['label'] == 'anomaly' and old['defect_ids'] == str(TYPE_ID[row['defect_type']])
        assert old['pixel_sha256'] not in forbidden_pixels, 'test/val pixel leak'
        approved.append(dict(old, usage='gen_train'))

    path = dest / f'{pcb}_split.csv'
    write_csv(path, sorted(approved, key=lambda r: r['file_path']))
    manifest_path = dest / 'manifest.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {
        'seed': 7017, 'shared_k': k, 'shared_split_seed': seed, 'split_sha256': {},
    }
    assert manifest['shared_k'] == k and manifest['shared_split_seed'] == seed
    manifest['split_sha256'][path.name] = sha(path)
    atomic_json(manifest_path, manifest)


def training_config(profile, checkpoint_root, data_root, splits, smoke=False):
    config = yaml.safe_load(Path(f'configs/ctransgan_{profile}.yaml').read_text())
    config.update(data_root=str(data_root), splits=str(splits), checkpoints=str(checkpoint_root))
    if smoke:
        config.update(require_cuda=False, device='cpu', profile='cpu_smoke')
        config['network'].update(
            noise_dim=8, encoder_width=16, encoder_depth=1, generator_width=64,
            generator_depths=[1, 1, 1], heads=2,
        )
        config['training'].update(
            classifier_steps=2, gan_steps=3, batch_size=2, gradient_accumulation=1,
            amp=False, checkpoint_every=1, log_every=1,
        )
    return config


def generate_board(config, state, splits, pcb, k, seed, per_type, out):
    dataset = PatchDataset(config['data_root'], config['splits'], pcb, 32, config['context_pixels'])
    plan = config['training']
    assert state['classifier_step'] == plan['classifier_steps'] and state['gan_step'] == plan['gan_steps']
    assert state['signature'] == training_signature(config, dataset)
    assert set(dataset.class_ids) == set(TYPE_ID.values())
    device = device_for(config)
    _, model, _ = build_models(config['network'], len(dataset.class_ids))
    model.load_state_dict(state['models']['generator'])
    model.to(device).eval()
    embeddings = state['embeddings'].to(device)
    normals = splits.normal_rows(pcb)
    gen_seed = condition_seed(METHOD, pcb, k, seed)
    rows = []

    for type_index, defect in enumerate(TYPES):
        class_id = TYPE_ID[defect]
        class_index = dataset.class_ids.index(class_id)
        candidates = [record for record in dataset.records if record['class_id'] == class_id]
        for i in range(per_type):
            rng = sample_rng(METHOD, pcb, k, seed, type_index * 100000 + i)
            source = candidates[int(rng.integers(len(candidates)))]
            normal = normals[int(rng.integers(len(normals)))]
            target, _ = splits.open(normal)
            source_row = dataset.by_path[source['source_anomaly']]
            assert target.size == (int(source_row['width']), int(source_row['height']))
            x0, y0, x1, y1 = source['box']
            support = np.asarray(dataset.component_mask(source)) > 0
            latent_seed = int(rng.integers(2**63 - 1))
            generator = torch.Generator(device='cpu').manual_seed(latent_seed)
            noise = torch.randn(1, config['network']['noise_dim'], generator=generator).to(device)
            with torch.inference_mode():
                patch = model(noise, embeddings[class_index:class_index + 1])[0]
            assert torch.isfinite(patch).all()
            pixels = ((patch.cpu().permute(1, 2, 0).numpy() + 1) * 127.5).round().clip(0, 255)
            resized = Image.fromarray(pixels.astype(np.uint8)).resize(
                (x1 - x0, y1 - y0), Image.Resampling.LANCZOS,
            )
            distance = distance_transform_edt(np.pad(support, 1))[1:-1, 1:-1]
            alpha = (np.minimum(1, distance / config['feather_pixels']) * support)[..., None]
            result = np.asarray(target).copy()
            region = np.s_[y0:y1, x0:x1]
            result[region] = np.rint(result[region] * (1 - alpha) + np.asarray(resized) * alpha)
            mask = np.zeros((target.height, target.width), dtype=np.uint8)
            mask[region] = support * class_id

            stem = f'{pcb}_{defect}_{i:03d}'
            image_path = out / 'images' / f'{stem}.jpg'
            mask_path = out / 'masks' / f'{stem}.png'
            Image.fromarray(result).save(image_path, quality=95)
            Image.fromarray(mask).save(mask_path)
            condition = {
                'checkpoint_signature': state['signature'], 'class_id': class_id,
                'component_id': source['component_id'], 'source_box': source['box'],
                'latent_seed': latent_seed, 'resize': False, 'mask': 'insertion support',
                'training_profile': config['profile'],
            }
            rows.append(dict(
                image=image_path.relative_to(PROJECT_ROOT).as_posix(),
                mask=mask_path.relative_to(PROJECT_ROOT).as_posix(),
                label=1, source_image=splits.prefix + source['source_anomaly'], score='',
                category=pcb, defect_type=defect, base_normal_image=normal['image'],
                gen_seed=gen_seed, condition=json.dumps(condition, sort_keys=True),
            ))
    return rows


def run(data_root, checkpoint_root, k=1, seed=0, profile='t4', per_type=40,
        shared_dir=None, audit_dir=None, smoke=False, pcbs=PCBS):
    assert per_type > 0
    case = f'K{k}_seed{seed}'
    work = Path(checkpoint_root) / profile / case
    if smoke:
        work = Path(checkpoint_root) / 'cpu_smoke' / case
    authorized = work / 'authorized'
    audit_dir = Path(audit_dir) if audit_dir else Path('splits')
    splits = SharedSplits(shared_dir, data_root)
    config = training_config(profile, work, data_root, authorized, smoke=smoke)
    method_dir = f'{METHOD}_smoke' if smoke else METHOD
    out = PROJECT_ROOT / 'Dataset' / 'generated' / method_dir / case
    for folder in ('images', 'masks'):
        (out / folder).mkdir(parents=True, exist_ok=True)
    rows = []
    for pcb in pcbs:
        restrict_training(splits, audit_dir, authorized, pcb, k, seed)
        state = train(config, pcb)
        rows.extend(generate_board(config, state, splits, pcb, k, seed, per_type, out))
        print(f'{case} {pcb}: {per_type * len(TYPES)} images, {state["dataset"]["real_anomaly_images"]} real sources', flush=True)
    manifest = out / 'manifest.csv'
    tmp = manifest.with_suffix('.tmp')
    with tmp.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    check_manifest(tmp, splits.splits)
    tmp.replace(manifest)
    print(check_manifest(manifest, splits.splits), flush=True)
    return manifest

In [ ]:
assert torch.cuda.is_available(), 'GPU 런타임을 연결한 뒤 이 셀을 다시 실행하세요.'
gpu_name = torch.cuda.get_device_name()
profile = 'a100' if 'A100' in gpu_name else 't4'
print(gpu_name, profile)

for k in K_VALUES:
    manifest = run(LOCAL_DATA, RUN / 'checkpoints/ctransgan', k=k, seed=SPLIT_SEED,
                   profile=profile, per_type=PER_TYPE)
    publish(manifest)

In [ ]:
from IPython.display import display

# 각 PCB의 첫 합성 예시를 Drive에서 확인합니다.
for pcb in PCBS:
    for method in ('copypaste', 'ctransgan'):
        manifest = RUN / f'Dataset/generated/{method}/K{K_VALUES[0]}_seed{SPLIT_SEED}/manifest.csv'
        row = next(r for r in read_rows(manifest) if r['category'] == pcb)
        with Image.open(RUN / row['image']) as image:
            preview = image.copy()
            preview.thumbnail((500, 375))
        print(pcb, method, row['defect_type'], row['source_image'])
        display(preview)